In [1]:
pip install pathlib tqdm datasets pandas soundfile librosa seaborn jiwer evaluate torch transformers numpy requests accelerate


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip uninstall -y datasets torchcodec
!pip install "datasets==3.6.0"

Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.16-py310-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
Using cached multiprocess-0.70.16-py310-none-any.whl (134 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.4.0
    Uninstalling fsspec-2026.4.0:
      Successfully uninstalled fsspec-2026.4.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocessm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [dill]
    Found existing 

In [2]:
import datasets
print(datasets.__version__)

3.6.0


In [3]:
import torch
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline,
)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from datasets import Dataset, DatasetDict, load_from_disk

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import re
import jiwer

In [4]:
device = "cuda:0"
torch_dtype = torch.float32
model_id = "openai/whisper-small"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
)
model.to(device)

# Configure generation
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
myst = load_from_disk(
    "/home/myst-v0.4.2/myst_dataset.ds"
)
myst

Loading dataset from disk:   0%|          | 0/23 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'speaker_id', 'date', 'time', 'session_type', 'version', 'split'],
        num_rows: 76924
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'speaker_id', 'date', 'time', 'session_type', 'version', 'split'],
        num_rows: 12238
    })
    test: Dataset({
        features: ['audio', 'transcription', 'speaker_id', 'date', 'time', 'session_type', 'version', 'split'],
        num_rows: 13169
    })
})

In [6]:
normalizer = BasicTextNormalizer()


def normalize_transcript(text):
    # The original transcript has annotations, for example a pause is <pau>
    # Remove tags in angle brackets
    text = re.sub(r"<[^>]*>", "", text)

    # These are "false starts" in the original transcript, for example th*
    # These are ignored by ASR
    # Remove words that end with asterisks (e.g., th*)
    text = re.sub(r"\S*\*", "", text)

    # Remove all punctuation
    normalized_text = normalizer(text)

    return normalized_text

In [10]:
def filter_short_long_samples(batch):
    audio_len = batch["audio"]["array"].shape[0] / batch["audio"]["sampling_rate"]
    return 5 <= audio_len <= 30


def filter_empty_transcripts(batch):
    return len(batch["transcription"].strip()) > 0


myst_filtered = myst.filter(filter_short_long_samples).filter(filter_empty_transcripts)

print(len(myst_filtered["train"]))

Filter:   0%|          | 0/76924 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12238 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13169 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36544 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6013 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6321 [00:00<?, ? examples/s]

36544


In [7]:
# Calculate lengths and get speaker IDs
data = []
for sample in myst_filtered["test"]:
    length = sample["audio"]["array"].shape[0] / sample["audio"]["sampling_rate"]
    data.append({"speaker_id": sample["speaker_id"], "length": length})

df = pd.DataFrame(data)

# Duration by sample
display(df["length"].describe())

# Sum duration per speaker
display(df.groupby("speaker_id")["length"].sum().describe())

NameError: name 'myst_filtered' is not defined

In [7]:
def prepare_dataset(batch):
    # Load audio
    audio = batch["audio"]
    sampling_rate = batch["audio"]["sampling_rate"]

    inputs = processor.feature_extractor(
        audio["array"],
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding="max_length",  # This ensures padding to max length in batch
        max_length=30 * sampling_rate,  # 30 seconds at 16kHz
        truncation=True,  # Truncate if longer than max_length
    )

    # Reprocess the filtered audio
    batch["input_features"] = inputs.input_features[0]

    # Normalize and encode target text
    normalized_text = normalize_transcript(batch["transcription"])
    batch["labels"] = processor.tokenizer(normalized_text, padding=True).input_ids

    return batch

In [ ]:
myst_filtered.save_to_disk(
    "/home/myst-v0.4.2/myst_fil.ds"
)

In [8]:
from datasets import load_from_disk

myst_filtered = load_from_disk(
    "/home/myst-v0.4.2/myst_filtered.ds"
)

In [15]:

myst_processed = myst_filtered.map(
    prepare_dataset,
    remove_columns=myst_filtered.column_names["train"],
    num_proc=1,
)

myst_processed["train"]

Map:   0%|          | 0/36544 [00:00<?, ? examples/s]

Map:   0%|          | 0/6013 [00:00<?, ? examples/s]

Map:   0%|          | 0/6321 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 36544
})

In [16]:
myst_processed.save_to_disk(
    "/home/myst-v0.4.2/myst_processed.ds"
)

Saving the dataset (0/71 shards):   0%|          | 0/36544 [00:00<?, ? examples/s]

Saving the dataset (0/12 shards):   0%|          | 0/6013 [00:00<?, ? examples/s]

Saving the dataset (0/13 shards):   0%|          | 0/6321 [00:00<?, ? examples/s]

In [9]:
myst_processed = load_from_disk(
    "/home/myst-v0.4.2/myst_processed.ds"
)

Loading dataset from disk:   0%|          | 0/71 [00:00<?, ?it/s]

In [10]:
def weighted_wer(ref: list[str], pred: list[str]):
    # Normalize both predictions and references
    pred_normalized = [normalize_transcript(text) for text in pred]
    label_normalized = [normalize_transcript(text) for text in ref]

    total_errors = 0
    total_words = 0

    for pred_text, ref_text in zip(pred_normalized, label_normalized):
        ref_words = ref_text.split()

        # Compute WER for this sample
        if len(ref_words) > 0:
            sample_wer = jiwer.wer(ref_text, pred_text)
        else:
            sample_wer = 0

        # Accumulate weighted errors
        sample_errors = sample_wer * len(ref_words)
        total_errors += sample_errors
        total_words += len(ref_words)

    weighted_wer = total_errors / total_words if total_words > 0 else 0.0

    return {"wer": weighted_wer}


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return weighted_wer(label_str, pred_str)

In [ ]:
def get_baseline(ds):
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        batch_size=64,
        device=device,
        chunk_length_s=30,
    )
    results = pipe(ds["test"]["audio"])
    return results


def get_wer(y_true, preds):
    y_pred = [d["text"] for d in preds]
    wer_score = weighted_wer(y_true, y_pred)
    return wer_score


preds = get_baseline(myst_filtered)
get_wer(myst_filtered["test"]["transcription"], preds)
# 0.19560642190320807

In [16]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [
            {"input_features": feature["input_features"]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [17]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/home/checkpoints/whisper-small-myst",
    per_device_train_batch_size=128,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=128,
    learning_rate=1e-6,
    warmup_steps=100,
    max_steps=1000,
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=False,
    dataloader_num_workers=8,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=50,
    predict_with_generate=True,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=myst_processed["train"],
    eval_dataset=myst_processed["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

In [18]:
# Start training
print("Starting training...")
trainer.train()

# 0.136113 with learning_rate 1e-6 on MyST

Starting training...


Step,Training Loss,Validation Loss,Wer
500,0.337046,0.441112,0.172587
1000,0.313131,0.430913,0.164809


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1000, training_loss=0.4975846652984619, metrics={'train_runtime': 1839.1648, 'train_samples_per_second': 69.597, 'train_steps_per_second': 0.544, 'total_flos': 3.688352284409856e+19, 'train_loss': 0.4975846652984619, 'epoch': 3.4965034965034967})

In [19]:
trainer.save_model("/home/checkpoints/myst1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
trainer.save_model("/home/whisper_finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Evaluate final model
print("Evaluating final model...")
final_results = trainer.evaluate()
print(f"Final WER: {final_results['eval_wer']:.4f}")

## Results Summary

### Dataset
- **Dataset:** MyST Children's Speech Corpus
- **Training samples:** 36,544
- **Validation samples:** 6,013
- **Test samples:** 6,321

### Model
- **Base model:** `openai/whisper-small`
- **Task:** Automatic Speech Recognition (ASR)
- **Framework:** Hugging Face Transformers (`Seq2SeqTrainer`)

### Training Configuration
| Parameter | Value |
|-----------|-------|
| GPU | NVIDIA H200 (141 GB VRAM) |
| Precision | BF16 |
| Learning Rate | `1e-6` |
| Warmup Steps | `100` |
| Max Training Steps | `1000` |
| Train Batch Size | `64` |
| Gradient Accumulation | `1` |
| Effective Batch Size | `64` |
| Evaluation Batch Size | `64` |
| Evaluation Strategy | Every 500 steps |
| Checkpoint Saving | Every 500 steps |

### Training Results

| Step | Training Loss | Validation Loss | WER |
|------|---------------:|----------------:|----:|
| 500 | 0.337046 | 0.441112 | **0.172587** |
| 1000 | 0.313131 | 0.430913 | **0.164809** |

### Final Performance

- **Final Word Error Rate (WER): `0.164809` (16.48%)**
- Validation loss decreased throughout training, indicating continued improvement without obvious signs of divergence.
- WER improved from **19%**(base) to **17.26%** at step 500 to **16.48%** at step 1000.

### Hardware Observations

Training was performed on an NVIDIA H200 GPU. During training:
- GPU compute utilization frequently reached **90–100%**.
- Power draw was close to the maximum (~690 W).
- GPU memory usage remained around **46 GB**, leaving significant VRAM headroom for experimenting with larger batch sizes or larger Whisper models in future experiments.

### Conclusion

Fine-tuning Whisper Small on the filtered MyST dataset successfully improved recognition performance, achieving a final **WER of 16.48%** after 1,000 training steps. The H200 GPU provided sufficient computational resources for efficient training, with substantial unused memory available for future scaling experiments.